# SpecDist — Colab Quickstart

**One-click pipeline: distil a Qwen3-0.6B draft from a Qwen3-8B teacher on a free T4.**

| Cell | What it does | Time |
|------|-------------|------|
| 1. Setup | Mount Drive · clone repo · install deps | ~3 min |
| 2. Authenticate | W&B + HuggingFace (from Colab Secrets) | ~30 s |
| 3. Run | Full pipeline (train → merge → eval) | ~4 h |
| 4. Resume | After session death — re-runs from last checkpoint | ~1 min + training |
| 5. Monitor | Check progress log + pipeline state any time | instant |
| 6. Dashboard | Live results dashboard in Colab tab | ~10 s |

---

### Prerequisites
1. **GPU runtime enabled**: Runtime → Change runtime type → T4 GPU (free) or A100 (Pro)
2. **Colab Secrets configured** (left sidebar → 🔑 Secrets):
   - `WANDB_API_KEY` — from https://wandb.ai/authorize
   - `HF_TOKEN` — from https://huggingface.co/settings/tokens (read access is enough)
3. Google Drive mounted for persistent checkpoints

> **Tip:** Run `--smoke` in Cell 3 first (~45 min) to verify the full code path before an overnight run.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1 — Setup
# Mount Google Drive (ALL artifacts live here — survive session restarts)
# Clone repo to ephemeral /content/ (fast, ~5 s; re-clones each session)
# Install dependencies
# ─────────────────────────────────────────────────────────────────────────────
import os, subprocess, sys

# ── 1a. Mount Drive ──────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# storage_root is the single directory that holds all persistent artifacts:
#   results.db, checkpoints/, logs/, pipeline_state_*.json
DRIVE_ROOT = "/content/drive/MyDrive/specdist"
os.makedirs(os.path.join(DRIVE_ROOT, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_ROOT, "logs"), exist_ok=True)
print(f"✓ All artifacts will persist at: {DRIVE_ROOT}")
print(f"    {DRIVE_ROOT}/results.db        ← experiment database")
print(f"    {DRIVE_ROOT}/checkpoints/      ← LoRA adapters")
print(f"    {DRIVE_ROOT}/logs/             ← pipeline + training logs")

# ── 1b. Clone repo ───────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR  = "/content/Distill-Spec-Research"
GBV_DIR   = f"{REPO_DIR}/gbv-research"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print("✓ Repo cloned")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("✓ Repo updated (already cloned)")

os.chdir(GBV_DIR)
print(f"✓ Working directory: {os.getcwd()}")

# ── 1c. Install Python dependencies ─────────────────────────────────────────
# bitsandbytes is required for 4-bit NF4 loading of the 8B teacher on a T4.
# accelerate is required by bitsandbytes device_map="auto".
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements.txt",
    "bitsandbytes",    # 4-bit NF4 teacher
    "accelerate",      # device_map support
    "flash-attn",      # optional — faster attention; skipped silently if compile fails
], check=False)  # flash-attn may fail on older drivers — that's OK
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements.txt",
    "bitsandbytes", "accelerate",
], check=True)   # re-run without flash-attn to ensure core deps are installed

print("✓ Dependencies installed")
print("\n--- Setup complete. Proceed to Cell 2 ---")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 — Authenticate (W&B + HuggingFace)
#
# Reads secrets from Colab Secrets (left sidebar → 🔑 icon).
# Required secrets:
#   WANDB_API_KEY  →  https://wandb.ai/authorize
#   HF_TOKEN       →  https://huggingface.co/settings/tokens
#
# If you prefer, paste keys directly below instead of using Secrets.
# ─────────────────────────────────────────────────────────────────────────────
import os

def _get_secret(name: str, fallback: str = "") -> str:
    """Read from Colab Secrets, then env, then fallback."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name, fallback)

# ── W&B ──────────────────────────────────────────────────────────────────────
WANDB_API_KEY = _get_secret("WANDB_API_KEY")
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    import wandb
    wandb.login(key=WANDB_API_KEY, relogin=True)
    print("✓ W&B authenticated")
else:
    print("⚠ WANDB_API_KEY not found — runs will be logged offline only.")
    print("  Add it via: left sidebar → 🔑 Secrets → + Add new secret")
    os.environ["WANDB_MODE"] = "offline"

# ── HuggingFace ──────────────────────────────────────────────────────────────
HF_TOKEN = _get_secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN   # older env var
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✓ HuggingFace Hub authenticated")
    except Exception as e:
        print(f"  [HF login] {e} — continuing anyway (public models don't need auth)")
else:
    print("ℹ HF_TOKEN not found — OK for public models (Qwen3-0.6B, Qwen3-8B).")

# ── Confirm GPU ───────────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}  "
          f"{free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")
    if total / 1024**3 < 12:
        print("⚠ Less than 12 GB VRAM detected — the colab config requires T4 (15 GB).")
        print("  Runtime → Change runtime type → T4 GPU")
else:
    print("⚠ No GPU detected — training will be extremely slow.")
    print("  Runtime → Change runtime type → T4 GPU")

print("\n--- Auth complete. Proceed to Cell 3 ---")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3 — Run the pipeline
#
# Config:  colab  →  Qwen3-0.6B draft,  Qwen3-8B teacher (4-bit NF4)
#          server →  Qwen3-0.6B draft,  Qwen3-8B teacher (bfloat16, needs A100)
#          laptop →  Qwen2.5-0.5B draft, Qwen3-0.6B teacher (test/debug)
#
# Modes:
#   --smoke              Fast smoke test (~45 min) — verifies code path before overnight
#   (no flag)            Full run  (~4 h on T4 with 500 steps per loss)
#   --losses kl,ebe      Run only specific distillation losses
#   --train_steps 1000   Override steps per loss from config
#
# Progress:  open Cell 5 in a separate tab to tail logs without interrupting this cell.
# Dashboard: run Cell 6 any time for a live results UI (works while training).
# ─────────────────────────────────────────────────────────────────────────────
import os, subprocess, sys

# storage_root: ALL artifacts (results.db, checkpoints, logs) go here.
# Must be on Google Drive so they survive session termination.
DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"

# ── Configuration ─────────────────────────────────────────────────────────────
CONFIG     = "colab"    # colab | server | laptop
SMOKE      = False      # True = quick smoke test first (~45 min)
LOSSES     = None       # None = all losses, or e.g. "kl,ebe" for subset
EXTRA_ARGS = []         # e.g. ["--train_steps", "1000"] to override steps

# ─────────────────────────────────────────────────────────────────────────────

# Safety check
if not os.path.isdir(DRIVE_ROOT):
    raise RuntimeError(
        f"Storage directory not found: {DRIVE_ROOT}\n"
        "Did you run Cell 1 (Setup) first?"
    )

os.chdir(GBV_DIR)

# Set env vars so every subprocess (train, eval, online) writes to Drive
os.environ["SPECDIST_STORAGE_ROOT"] = DRIVE_ROOT
os.environ["SPECDIST_DB_PATH"]       = os.path.join(DRIVE_ROOT, "results.db")
os.environ["SPECDIST_LOGS_ROOT"]     = os.path.join(DRIVE_ROOT, "logs")

pipeline_args = [
    sys.executable, "orchestration/experiment.py",
    "--config",       CONFIG,
    "--storage_root", DRIVE_ROOT,  # pins DB + checkpoints + logs to Drive
    "--yes",                       # skip interactive prompts
]
if SMOKE:
    pipeline_args.append("--smoke")
    print("Running SMOKE TEST (n=5, 50 steps — verifies code path, ~45 min)")
else:
    print(f"Running FULL pipeline ({CONFIG} config)")
if LOSSES:
    pipeline_args += ["--losses", LOSSES]
    print(f"  Losses: {LOSSES}")
pipeline_args += EXTRA_ARGS

print(f"  Storage root → {DRIVE_ROOT}")
print(f"  Results DB   → {DRIVE_ROOT}/results.db")
print(f"  Checkpoints  → {DRIVE_ROOT}/checkpoints/")
print(f"  Live logs    → {DRIVE_ROOT}/logs/pipeline_output.log  (Cell 5)")
print(f"  W&B project  → distillspec")
print()

# Run (output streams live to the notebook cell)
result = subprocess.run(pipeline_args, cwd=GBV_DIR)

if result.returncode == 0:
    print("\n✓ Pipeline complete!")
    print(f"  Results DB  : {DRIVE_ROOT}/results.db")
    print(f"  Checkpoints : {DRIVE_ROOT}/checkpoints/")
    print(f"  Dashboard   : run Cell 6 to explore results")
else:
    print(f"\n✗ Pipeline exited with code {result.returncode}")
    print("  Check the output above for [OOM] / [ERROR] / Traceback.")
    print("  To resume: re-run this cell — the pipeline skips completed steps automatically.")
    print(f"  Full log  : {DRIVE_ROOT}/logs/pipeline_output.log  (Cell 5 → tail it)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4 — Resume after session death
#
# Colab kills sessions after ~90 min idle (free) or ~12 h (Pro).
# Run this cell to continue from the last saved checkpoint.
# The pipeline auto-detects completed steps via checkpoint files on Drive.
# ─────────────────────────────────────────────────────────────────────────────
import os, subprocess, sys

# ── Re-mount Drive and re-clone if needed (idempotent) ───────────────────────
from google.colab import drive
drive.mount('/content/drive')

REPO_URL   = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR   = "/content/Distill-Spec-Research"
GBV_DIR    = f"{REPO_DIR}/gbv-research"
DRIVE_ROOT = "/content/drive/MyDrive/specdist"   # same as Cell 1 & 3

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", f"{GBV_DIR}/requirements.txt",
                    "bitsandbytes", "accelerate"], check=True)
    print("✓ Repo re-cloned, deps re-installed")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("✓ Repo still present — updated")

os.chdir(GBV_DIR)

# ── Re-authenticate W&B ───────────────────────────────────────────────────────
try:
    from google.colab import userdata
    key = userdata.get("WANDB_API_KEY")
    if key:
        os.environ["WANDB_API_KEY"] = key
        import wandb; wandb.login(key=key, relogin=True)
        print("✓ W&B re-authenticated")
except Exception:
    os.environ.setdefault("WANDB_MODE", "offline")
    print("⚠ W&B key not found — logging offline")

# ── Re-set storage env vars ───────────────────────────────────────────────────
os.environ["SPECDIST_STORAGE_ROOT"] = DRIVE_ROOT
os.environ["SPECDIST_DB_PATH"]       = os.path.join(DRIVE_ROOT, "results.db")
os.environ["SPECDIST_LOGS_ROOT"]     = os.path.join(DRIVE_ROOT, "logs")

# ── Resume pipeline ───────────────────────────────────────────────────────────
CONFIG = "colab"   # must match what was used in Cell 3

print(f"\nResuming pipeline (config={CONFIG}, storage_root={DRIVE_ROOT})")
print("Pipeline will skip already-completed steps and resume from last failure.\n")

subprocess.run([
    sys.executable, "orchestration/experiment.py",
    "--config",       CONFIG,
    "--storage_root", DRIVE_ROOT,
    "--yes",
], cwd=GBV_DIR)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5 — Monitor progress  (safe to run any time, even while Cell 3 is running)
#
# Shows:
#   • Last 60 lines of the live pipeline log (Drive-persisted)
#   • Which steps are done / running / pending (from pipeline state file)
#   • Any per-step error snapshots if a step crashed
# ─────────────────────────────────────────────────────────────────────────────
import os, json, glob, datetime

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
LOG_FILE   = f"{DRIVE_ROOT}/logs/pipeline_output.log"
STATE_FILE = f"{DRIVE_ROOT}/pipeline_state_colab.json"

# ── 1. Pipeline state ─────────────────────────────────────────────────────────
print("=" * 60)
print("PIPELINE STATE")
print("=" * 60)
if os.path.exists(STATE_FILE):
    state = json.load(open(STATE_FILE, encoding="utf-8"))
    steps = state.get("steps", {})
    counts = {"done": 0, "running": 0, "pending": 0, "error": 0}
    for sid, info in steps.items():
        s = info.get("status", "pending")
        counts[s] = counts.get(s, 0) + 1
        icon = {"done": "✓", "running": "▶", "error": "✗"}.get(s, "·")
        ts = info.get("finished_at") or info.get("started_at") or ""
        print(f"  {icon} {sid:45s}  {s}  {ts[:16]}")
    print(f"\n  Done: {counts['done']}  Running: {counts['running']}  Pending: {counts.get('pending',0)}  Error: {counts['error']}")
else:
    print(f"  State file not found: {STATE_FILE}")
    print("  Pipeline hasn't started yet, or Drive isn't mounted.")

# ── 2. Live log tail ─────────────────────────────────────────────────────────
print()
print("=" * 60)
print("PIPELINE LOG (last 60 lines)")
print("=" * 60)
if os.path.exists(LOG_FILE):
    lines = open(LOG_FILE, encoding="utf-8", errors="replace").readlines()
    print("".join(lines[-60:]))
    mtime = os.path.getmtime(LOG_FILE)
    age_s = datetime.datetime.now().timestamp() - mtime
    status = "● ACTIVE" if age_s < 120 else f"⚠ last update {int(age_s//60)} min ago"
    print(f"[Log: {LOG_FILE}  |  {len(lines)} lines  |  {status}]")
else:
    print(f"  Log not found: {LOG_FILE}")
    print("  Pipeline hasn't started writing logs yet.")

# ── 3. Error snapshots ────────────────────────────────────────────────────────
err_logs = sorted(glob.glob(f"{DRIVE_ROOT}/logs/step_*_error.log"))
if err_logs:
    print()
    print("=" * 60)
    print(f"ERROR SNAPSHOTS ({len(err_logs)} file(s))")
    print("=" * 60)
    for f in err_logs:
        print(f"\n--- {os.path.basename(f)} ---")
        content = open(f, encoding="utf-8", errors="replace").read()
        # Print last 2000 chars of each error log (the traceback is usually at the end)
        print(content[-2000:] if len(content) > 2000 else content)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6 — Live results dashboard
#
# Starts the Flask dashboard server bound to 0.0.0.0 so Colab's port proxy
# can tunnel it to a public HTTPS URL you can open in any browser tab.
#
# Safe to run while Cell 3 is training — the dashboard reads the DB live
# and shows new eval rows as they land.
#
# Shows:
#   • Block efficiency heatmap (loss × verifier)
#   • Training loss curves
#   • Per-step acceptance-rate breakdown
#   • Live pipeline log tail
#
# To stop: Runtime → Interrupt execution  (or just close the tab — server
# keeps running in background until the Colab session dies)
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, time, threading, subprocess
from google.colab.output import eval_js

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"
PORT       = 5000

# Point dashboard at Drive DB + logs (not ephemeral /content/)
os.environ["SPECDIST_DB_PATH"]   = f"{DRIVE_ROOT}/results.db"
os.environ["SPECDIST_LOGS_ROOT"] = f"{DRIVE_ROOT}/logs"

if not os.path.exists(os.environ["SPECDIST_DB_PATH"]):
    print(f"⚠ No results DB yet at {os.environ['SPECDIST_DB_PATH']}")
    print("  Run Cell 3 first to generate some results, then come back here.")
else:
    # Launch Flask server in a background thread so this cell doesn't block
    _server_proc = [None]

    def _start_server():
        _server_proc[0] = subprocess.Popen(
            [
                sys.executable,
                f"{GBV_DIR}/dashboard/training_dashboard.py",
                "--host", "0.0.0.0",
                "--port", str(PORT),
            ],
            env=os.environ.copy(),
        )
        _server_proc[0].wait()

    t = threading.Thread(target=_start_server, daemon=True)
    t.start()

    # Give Flask a moment to bind the port
    time.sleep(3)

    # Get the Colab-proxied public URL for this port
    public_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")

    print(f"✓ Dashboard running")
    print(f"  Open this URL in any browser tab:")
    print(f"  {public_url}")
    print()
    print(f"  DB   : {os.environ['SPECDIST_DB_PATH']}")
    print(f"  Logs : {os.environ['SPECDIST_LOGS_ROOT']}")
    print(f"  Port : {PORT} (proxied by Colab to HTTPS above)")
    print()
    print("  The dashboard auto-refreshes every 15 s — new eval rows appear as they land.")
    print("  To stop the server: Runtime → Interrupt execution")